<a href="https://colab.research.google.com/github/Mogi21/pytorch-deep-learning-labs/blob/main/06_pytorch_transformers.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from typing import List, Optional

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset
from torch.optim import AdamW
from transformers import (
    AutoTokenizer,
    AutoModel,
    AutoModelForSequenceClassification,
    AutoModelForMaskedLM
)
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.metrics import f1_score

# ==========================================
# 1. CONFIGURATION & DEVICE SETUP
# ==========================================
def set_seed(seed: int = 42) -> None:
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed()
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Executing NLP pipeline on device: {device}")

# ==========================================
# 2. CONTEXTUAL EMBEDDINGS & TEXT SIMILARITY
# ==========================================
print("\n--- Initializing BERT for Contextual Embeddings ---")
# Load base BERT model and tokenizer for feature extraction
bert_tokenizer = AutoTokenizer.from_pretrained('bert-base-uncased')
bert_model = AutoModel.from_pretrained('bert-base-uncased').to(device)
bert_model.eval()

def compute_text_similarity(texts: List[str], model: AutoModel, tokenizer: AutoTokenizer, device: torch.device) -> np.ndarray:
    """
    Computes pairwise cosine similarity between a list of texts using
    pre-trained transformer pooler embeddings.
    """
    inputs = tokenizer(texts, padding=True, truncation=True, return_tensors='pt').to(device)

    with torch.no_grad():
        outputs = model(**inputs)

    # Extract the [CLS] token representations (pooler_output) for sentence-level embeddings
    embeddings = outputs.pooler_output.cpu().numpy()

    # Compute pairwise cosine similarity matrix
    similarity_matrix = cosine_similarity(embeddings)
    return similarity_matrix

# Example Usage
sample_texts = [
    "Paul is cooking dinner for his friend.",
    "Maria is cooking dinner for her friend.",
    "Stockholm is a beautiful city!"
]
similarity_matrix = compute_text_similarity(sample_texts, bert_model, bert_tokenizer, device)
print("Similarity Matrix computed successfully.")

# ==========================================
# 3. FINE-TUNING TRANSFORMERS (RoBERTa)
# ==========================================
print("\n--- Initiating Fine-Tuning Pipeline (RoBERTa) ---")

class TextClassificationDataset(Dataset):
    """Custom PyTorch Dataset for handling raw text files and labels."""
    def __init__(self, file_paths: List[str], labels: List[int], tokenizer, max_length: int = 128):
        self.file_paths = file_paths
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.file_paths)

    def __getitem__(self, idx):
        with open(self.file_paths[idx], 'r', encoding='utf-8') as f:
            text = f.read()

        encoding = self.tokenizer(
            text,
            truncation=True,
            padding='max_length',
            max_length=self.max_length,
            return_tensors='pt'
        )

        return {
            'input_ids': encoding['input_ids'].flatten(),
            'attention_mask': encoding['attention_mask'].flatten(),
            'labels': torch.tensor(self.labels[idx], dtype=torch.long)
        }

# Setup RoBERTa specifically for Sequence Classification
roberta_model_name = 'roberta-base'
roberta_tokenizer = AutoTokenizer.from_pretrained(roberta_model_name)
roberta_model = AutoModelForSequenceClassification.from_pretrained(roberta_model_name, num_labels=2).to(device)

def train_transformer(model: nn.Module, train_loader: DataLoader, epochs: int = 2, lr: float = 2e-5):
    """Fine-tunes a pre-trained transformer model using AdamW."""
    optimizer = AdamW(model.parameters(), lr=lr)

    for epoch in range(epochs):
        model.train()
        epoch_loss = 0.0

        for batch in train_loader:
            optimizer.zero_grad()

            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['labels'].to(device)

            outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
            loss = outputs.loss

            loss.backward()
            optimizer.step()

            epoch_loss += loss.item()

        print(f"Epoch {epoch+1}/{epochs} | Training Loss: {epoch_loss/len(train_loader):.4f}")

# Note: To run this segment, initialize the `TextClassificationDataset` with your actual
# file paths and pass it to a DataLoader, then call `train_transformer()`.

# ==========================================
# 4. AUTOREGRESSIVE GENERATION VIA MASKED LM
# ==========================================
print("\n--- Loading Masked Language Model for Autoregressive Generation ---")
# BERT is traditionally bidirectional, but we can simulate autoregressive
# generation by iteratively predicting the [MASK] token.
mlm_model = AutoModelForMaskedLM.from_pretrained('bert-base-uncased').to(device)
mlm_model.eval()

def generate_text_autoregressive(prompt: str, max_tokens: Optional[int] = None,
                                 model=mlm_model, tokenizer=bert_tokenizer, device=device) -> str:
    """
    Simulates causal autoregressive generation using a Masked Language Model.
    Iteratively predicts the next token by appending a [MASK] token to the sequence.
    """
    if max_tokens is None:
        max_tokens = tokenizer.model_max_length - len(tokenizer(prompt).input_ids)

    # Pad prompt with [MASK] tokens
    prompt += ' '.join(['[MASK]'] * max_tokens)
    inputs = tokenizer(prompt, return_tensors="pt").to(device)

    with torch.no_grad():
        logits = model(**inputs).logits

    # Isolate the logits for the first masked token
    predicted_token_id = logits[0, -max_tokens-1].argmax(axis=-1).cpu().tolist()
    text = tokenizer.decode(inputs.input_ids[0, 1:-max_tokens-1].cpu().tolist() + [predicted_token_id])

    # Base conditions to halt generation
    if max_tokens == 1:
        return text
    if predicted_token_id == tokenizer.vocab['.']:
        return text
    if predicted_token_id == tokenizer.sep_token_id:
        return text

    # Recursively predict the next token
    return generate_text_autoregressive(text, max_tokens=max_tokens-1, model=model, tokenizer=tokenizer, device=device)

# Example Usage
prompt = "The capital of France is"
generated_output = generate_text_autoregressive(prompt, max_tokens=5)
print(f"Prompt: '{prompt}'")
print(f"Generated Output: '{generated_output}'")

Executing NLP pipeline on device: cpu

--- Initializing BERT for Contextual Embeddings ---


config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

...

[transformers] RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                        | Status     | 
---------------------------+------------+-
lm_head.dense.bias         | UNEXPECTED | 
lm_head.dense.weight       | UNEXPECTED | 
lm_head.layer_norm.bias    | UNEXPECTED | 
lm_head.bias               | UNEXPECTED | 
lm_head.layer_norm.weight  | UNEXPECTED | 
classifier.out_proj.bias   | MISSING    | 
classifier.dense.weight    | MISSING    | 
classifier.dense.bias      | MISSING    | 
classifier.out_proj.weight | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.



--- Loading Masked Language Model for Autoregressive Generation ---


Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

[transformers] BertForMaskedLM LOAD REPORT from: bert-base-uncased
Key                         | Status     |  | 
----------------------------+------------+--+-
bert.pooler.dense.weight    | UNEXPECTED |  | 
cls.seq_relationship.weight | UNEXPECTED |  | 
bert.pooler.dense.bias      | UNEXPECTED |  | 
cls.seq_relationship.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Prompt: 'The capital of France is'
Generated Output: 'the capital of france is the paris region capital.'
